# Verificación — Spark lee tu datalake

**Curso:** ST1630-2026-2 · **Semana:** S4-S5
**Estudiantes:** Hellen Yanes Doria, Sebastian Salazar Henao, Andres Velez Alvarez, Samuel Samper Cardona
**Fecha:** 13/08/2026

## Objetivo

Cerrar el Lab 1a confirmando que tu clúster EMR puede leer el datalake
que construiste (Partes 1-4): conectar Spark a tu bucket S3, leer el
archivo Parquet de Bronze, y repetir el benchmark Parquet vs. CSV visto
en la clase de S4.

**Qué debe verse al final para confirmar que el lab está completo:**
- La Celda 2 muestra el schema y 5 filas del Parquet leído desde S3
  (si esto funciona, tu bucket, tu rol IAM y tu clúster están bien
  configurados de punta a punta).
- La Celda 3 imprime el tiempo de una misma consulta en Parquet y en
  CSV, y el ratio entre ambos.
- Completaste el análisis de la Celda 4 y capturaste el DAG de Spark UI
  como indica la Celda 5.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# getOrCreate() funciona tanto en EMR (donde ya existe una sesión activa
# administrada por el clúster) como en un entorno local con pyspark
# instalado, sin necesitar ramas de código distintas.
spark = SparkSession.builder.appName("ST1630-Lab1a-Verificacion").getOrCreate()

# EDITAR: reemplaza por el bucket que creaste en setup_s3.sh
# (convención: st1630-{tu-usuario}-{año})
BUCKET = "st1630-ssamperc-2026"

# En EMR, S3 se referencia directamente con el esquema s3://.
# En local (con las credenciales de AWS Academy exportadas), la misma
# ruta también funciona porque Spark usa el conector S3A por debajo.
ruta_parquet = f"s3://{BUCKET}/bronze/ventas/prueba_parquet.parquet"
ruta_csv = f"s3://{BUCKET}/bronze/ventas/prueba_csv.csv"

df_parquet = spark.read.parquet(ruta_parquet)

df_parquet.printSchema()
df_parquet.show(5, truncate=False)

# Si ves el schema y las filas de arriba, tu datalake funciona
# correctamente de punta a punta: bucket, permisos IAM y clúster EMR.
print("Filas leídas:", df_parquet.count())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/13 23:44:40 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


root
 |-- order_id: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- region: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- cantidad: long (nullable = true)
 |-- precio_unit: double (nullable = true)
 |-- total: double (nullable = true)
 |-- canal: string (nullable = true)
 |-- devuelto: boolean (nullable = true)



+----------+----------+------------+---------+-----------+--------+-----------+---------+------+--------+
|order_id  |fecha     |region      |producto |categoria  |cantidad|precio_unit|total    |canal |devuelto|
+----------+----------+------------+---------+-----------+--------+-----------+---------+------+--------+
|ORD-000001|2026-04-22|Cali        |Mouse    |Electrónica|1       |789300.0   |789300.0 |online|false   |
|ORD-000002|2026-03-22|Barranquilla|Gorra    |Ropa       |2       |57000.0    |114000.0 |tienda|false   |
|ORD-000003|2026-01-06|Bogotá      |Zapatos  |Ropa       |4       |163800.0   |655200.0 |online|false   |
|ORD-000004|2025-11-26|Barranquilla|Panela   |Alimentos  |3       |54900.0    |164700.0 |online|false   |
|ORD-000005|2026-01-07|Cali        |Audífonos|Electrónica|4       |251100.0   |1004400.0|tienda|true    |
+----------+----------+------------+---------+-----------+--------+-----------+---------+------+--------+
only showing top 5 rows

Filas leídas: 10000


In [2]:
import time

df_csv = spark.read.option("header", "true").option("inferSchema", "true").csv(ruta_csv)

# Misma consulta sobre ambos formatos: filtrar por región y categoría,
# y agregar el total vendido -- el tipo de consulta selectiva que se
# beneficia de predicado pushdown y column pruning en formatos
# columnares (visto en la clase de S4, slide de Parquet vs. CSV).

def benchmark(df, nombre):
    inicio = time.time()
    resultado = (
        df.filter((F.col("region") == "Bogotá") & (F.col("categoria") == "Electrónica"))
          .groupBy("producto")
          .agg(F.sum("total").alias("total_vendido"))
          .orderBy(F.col("total_vendido").desc())
          .collect()  # acción -- fuerza la ejecución real, no solo el plan
    )
    duracion = time.time() - inicio
    print(f"{nombre}: {duracion:.3f} s ({len(resultado)} filas de resultado)")
    return duracion

tiempo_parquet = benchmark(df_parquet, "Parquet")
tiempo_csv = benchmark(df_csv, "CSV")

ratio = tiempo_csv / tiempo_parquet if tiempo_parquet > 0 else float("inf")
print(f"\nRatio CSV / Parquet: {ratio:.2f}x")

Parquet: 1.519 s (6 filas de resultado)
CSV: 0.821 s (6 filas de resultado)

Ratio CSV / Parquet: 0.54x


## Análisis — completa antes de entregar

### a) Tamaño en disco
`prueba_parquet.parquet`: 185.4 KB · `prueba_csv.csv`: 788.5 KB.
El CSV pesa ~4.25x más que el Parquet, pese a contener los mismos datos, por la compresión columnar de Parquet.

### b) Tiempo de la consulta
Parquet: 1.519 s · CSV: 0.821 s (Celda 3).

### c) Ratio de mejora
Ratio CSV/Parquet = 0.54x — el CSV fue más rápido, al contrario de lo esperado (~9x a favor de Parquet visto en clase). Con solo 10,000 filas en un único archivo sin particionar, el dataset cabe fácilmente en memoria y las ventajas de Parquet (column pruning, predicate pushdown, menos I/O) casi no se notan. Pesa más el overhead fijo: la lectura del Parquet probablemente pagó el costo de "cold start" del conector S3A y la inicialización del plan de Spark, mientras el CSV se benefició de un clúster ya "caliente". Con un dataset más grande o particionado, el resultado debería revertirse a favor de Parquet.

### d) Conexión con el Teorema CAP
S3 con replicación multi-AZ es una decisión **CP**: prioriza Consistencia (lecturas siempre ven el último write confirmado, gracias a "strong read-after-write consistency") y Tolerancia a Particiones (sigue funcionando aunque una AZ quede aislada). Sacrifica Disponibilidad en el caso extremo: ante una partición de red entre AZs, S3 puede rechazar o demorar una operación en vez de devolver datos potencialmente desactualizados.

## Captura del DAG en Spark UI

1. En EMR Studio (o en la consola de tu clúster), abre **Spark UI /
   History Server**.
2. Busca el job correspondiente a la Celda 3 (el `groupBy` + `agg` +
   `orderBy` sobre el Parquet).
3. Abre la pestaña **SQL / DataFrame** y captura una imagen del plan
   (o del DAG visual) que incluya al menos un nodo **Exchange**.
4. Guarda la captura como `dag_spark_ui.png` dentro de tu carpeta de
   entrega y referencíala en tu PR.

**Verifica:** la captura debe mostrar el nombre de tu aplicación
(`ST1630-Lab1a-Verificacion`, definido en la Celda 2) para que quede
claro que es tu propia ejecución.

## Bitácora de delegación

Completa según lo que realmente delegaste a un agente de IA durante
este notebook (ver `../../../docs/politica-ia.md`).

| Tarea | ¿Delegado a agente? | Herramienta | Justificación |
|---|---|---|---|
| Boilerplate de SparkSession / lectura de S3 | → [sí/no/parcial] | → [herramienta] | → [tu justificación aquí] |
| Diseño de la consulta del benchmark (Celda 3) | → [sí/no/parcial] | → [herramienta] | → [tu justificación aquí] |
| Interpretación de los resultados (Celda 4) | → [sí/no/parcial] | → [herramienta] | → [tu justificación aquí] |
| Troubleshooting de errores de conexión a S3 | → [sí/no/parcial] | → [herramienta] | → [tu justificación aquí] |

> Recuerda: la interpretación de los resultados y la conexión con CAP
> (pregunta d) deben reflejar tu propio razonamiento — ver
> `../README.md`, sección "Bitácora de delegación".